# DA6401 Assignment 3 — Transformer NMT

**Before running:**
1. Enable GPU: Settings → Accelerator → **T4 GPU**
2. Add secrets (Add-ons → Secrets):
   - `WANDB_API_KEY` → your key from wandb.ai/authorize
   - `GITHUB_TOKEN`  → your GitHub personal access token (if repo is private)

## Cell 1 — Install dependencies

In [ ]:
%%capture
!pip install wandb sacrebleu seaborn gdown datasets spacy
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm

## Cell 2 — Clone repo from GitHub

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

GITHUB_USER = "Anurag9Dhiman"
GITHUB_REPO = "da6401_assignment_3"

try:
    token = secrets.get_secret("GITHUB_TOKEN")
    clone_url = f"https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
except Exception:
    clone_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    print("No GITHUB_TOKEN found — cloning as public repo")

!git clone {clone_url} /kaggle/working/project
os.chdir("/kaggle/working/project")
print(f"\nWorking directory: {os.getcwd()}")
!ls

## Cell 3 — Authenticate W&B

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)
print("W&B login successful")

## Cell 4 — Verify GPU

In [ ]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 5 — Run training + all W&B experiments (~1 hour)

In [ ]:
from train import run_training_experiment

run_training_experiment()

## Cell 6 — Upload checkpoint to Google Drive

After training, `best_checkpoint.pt` is in `/kaggle/working/project/`.  
Upload it to Google Drive, then paste the file ID into `_GDRIVE_FILE_ID` in `model.py`.

In [ ]:
import os

ckpt = "/kaggle/working/project/best_checkpoint.pt"
print(f"Checkpoint exists : {os.path.exists(ckpt)}")
print(f"Size              : {os.path.getsize(ckpt) / 1e6:.1f} MB")
print()
print("Next steps:")
print("  1. Kaggle notebook → Output tab → Download best_checkpoint.pt")
print("  2. Upload to Google Drive")
print("  3. Right-click file → Get link → copy file ID from URL")
print('  4. Paste into model.py:  _GDRIVE_FILE_ID = "paste_here"')
print("  5. git add model.py && git commit && git push")

## Cell 7 — Sanity check: test `infer()` exactly like the autograder

Run this **after** filling `_GDRIVE_FILE_ID` in `model.py` and pushing to GitHub.

In [ ]:
# Pull latest model.py (with the real file ID) from GitHub
!git pull

import importlib, sys
# Remove cached modules so the updated model.py is re-imported
for mod in ["model", "train", "dataset", "lr_scheduler"]:
    sys.modules.pop(mod, None)

import torch
from model import Transformer

device = "cuda" if torch.cuda.is_available() else "cpu"

# Exactly what the autograder does
model = Transformer().to(device)
model.eval()

test_sentences = [
    "Ein Mann sitzt auf einer Bank.",
    "Eine Frau liest ein Buch.",
    "Kinder spielen im Park.",
]

for de in test_sentences:
    en = model.infer(de)
    print(f"DE: {de}")
    print(f"EN: {en}")
    print()